# 03 · Least squares and why naive inversion fails

We now attempt our first inversion: given noisy surface measurements, recover
the slip that produced them. The plan sounds simple. Chapter 02 gave us
$\mathbf{d} = G\mathbf{m}$, so we will ask a solver for the $\mathbf{m}$ that
fits the data best. This chapter carries that plan out honestly, and it fails
in a spectacular and instructive way. Understanding *why* it fails is the key
to everything in Chapter 04.

**Learning objectives**

By the end of the chapter, you will be able to:

- write the linear inverse problem with an explicit noise term
- explain ordinary and weighted least squares in words and symbols
- build a synthetic test in which the true slip is known
- run an unregularized inversion with `geodef.solve`
- interpret reduced chi-squared, and state what it cannot tell you
- recognize overfitting and connect it to the condition number of $G$

**Prerequisites:** Chapters 01 and 02; the least-squares idea from Chapter 00
**Estimated time:** 60–90 minutes

GeoDef's axes, signs, and units are summarized in
[the conventions guide](../docs/conventions.md). New terms are introduced in
words here and collected in [the glossary](../docs/glossary.md).

## 1. The inverse problem, stated honestly

Real measurements are never exactly the forward prediction. GNSS positions
wobble with atmospheric delays, antenna effects, and processing choices. An
honest statement of the problem therefore includes a noise term:

$$ \mathbf{d} = G\,\mathbf{m} + \boldsymbol{\varepsilon}. $$

Here $\mathbf{d}$ holds the measurements, $G$ is the Green's matrix from
Chapter 02, $\mathbf{m}$ is the slip we want, and $\boldsymbol{\varepsilon}$
is the measurement noise. We do not know the individual noise values; we only
know something about their typical size, summarized by each measurement's
reported uncertainty $\sigma_i$.

That one extra term changes the goal. We are no longer solving an equation
exactly; we are estimating $\mathbf{m}$ from data that are partly signal and
partly noise. A good method must fit the signal without also fitting the
noise.

## 2. Ordinary least squares

**Ordinary least squares** chooses the model whose predictions come closest to
the data, measured by the total squared misfit
$\lVert G\mathbf{m} - \mathbf{d}\rVert^2$, exactly as in the line fit of
Chapter 00. Calculus turns "make this as small as possible" into a solvable
equation: at the minimum, the gradient of the misfit is zero, which leads to
the **normal equations**

$$ G^{\mathsf T} G\, \hat{\mathbf{m}} = G^{\mathsf T}\mathbf{d}. $$

The hat on $\hat{\mathbf{m}}$ marks an *estimate* built from noisy data,
rather than the true slip. $G^{\mathsf T}$ is the transpose from Chapter 00,
and $G^{\mathsf T} G$ is a square matrix, so this is a system a computer can
solve.

## 3. Weighted least squares

Ordinary least squares treats every observation as equally believable. Real
datasets are not like that: a vertical GNSS component may be several times
noisier than a horizontal one. **Weighted least squares** divides each
residual by that observation's uncertainty before squaring, so a 5 mm misfit
counts heavily for a precise measurement and lightly for a noisy one.

Collecting the uncertainties into the data covariance matrix $C_d$ and writing
$W = C_d^{-1}$ for the weights, the estimate becomes

$$ \hat{\mathbf{m}} = \left(G^{\mathsf T} W G\right)^{-1} G^{\mathsf T} W \mathbf{d}. $$

When the noise model is correct, this weighted estimate is the best possible
linear, unbiased one (a classical result called the Gauss-Markov theorem).
GeoDef builds $W$ from each dataset's stated uncertainties automatically, so
`geodef.solve` performs weighted least squares by default; you never form $W$
yourself.

## 4. Where the plan goes wrong

The formula above contains the inverse of $G^{\mathsf T} W G$, and Chapter 02
warned us about this moment. When two patches produce nearly the same surface
pattern, their columns of $G$ are nearly parallel, and $G^{\mathsf T} W G$
becomes **ill-conditioned**: technically invertible, but barely.

An ill-conditioned inverse is an amplifier. Tiny changes in the data, such as
a centimeter of measurement noise, are magnified into huge changes in the
estimated slip. The amplified noise takes a characteristic shape: large
positive slip on one patch canceled by large negative slip on its neighbor,
because those alternating patterns are precisely the ones the surface data
almost cannot see.

(For readers who enjoy linear algebra: the singular value decomposition makes
this precise. Each independent pattern of slip has a singular value measuring
how strongly it shows up in the data, and the least-squares solution *divides*
by those values, so patterns with tiny singular values are where noise
explodes. The further reading at the end develops this view.)

## 5. A synthetic test scenario

The fair way to test an inversion method is a **synthetic test**: invent a
true slip model, generate the data it would produce, add realistic noise, and
then check whether the method recovers the truth. With real data we never get
to make that comparison.

This scenario, a megathrust with a single smooth patch of slip, is reused
through Chapter 10, so every method faces the same target. We build it in four
short steps: the fault, the true slip, the stations, and the noisy data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

%matplotlib inline

rng = np.random.default_rng(0)   # seeded so every run draws the same noise

The fault is the shallow-dipping megathrust of Chapter 01, now divided into 12
patches along strike and 6 down dip.

In [ ]:
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,
    strike=315.0, dip=15.0,
    length=180e3, width=90e3,
    n_length=12, n_width=6,
)
N = fault.n_patches
n_length, n_width = fault.grid_shape
print(f"{N} patches ({n_length} along strike x {n_width} down dip)")

The true slip is a smooth bell-shaped thrust patch, 3 m at its peak, placed
near the middle of the fault. As in Chapter 01, the model vector is blocked as
`[strike-slip | dip-slip]`, and the strike-slip half is zero.

In [ ]:
along = np.arange(N) % n_length          # along-strike position of each patch
down = np.arange(N) // n_length          # down-dip position of each patch
center_along = (n_length - 1) / 2
center_down = n_width / 2
bump = np.exp(-(((along - center_along) / 3.0) ** 2
                + ((down - center_down) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

The observing network is an 8 by 8 grid of GNSS stations. Note the argument
order convention: GeoDef datasets take longitude first, then latitude.

In [ ]:
grid_lon, grid_lat = np.meshgrid(np.linspace(98.5, 101.5, 8),
                                 np.linspace(-3.6, -0.4, 8))
station_lon = grid_lon.ravel()
station_lat = grid_lat.ravel()
n_stations = station_lon.size

Finally, the synthetic data: forward-model the true slip with the Green's
matrix, then add seeded Gaussian noise with a 1 cm standard deviation to every
component. The dataset records that same 1 cm as its uncertainty, so in this
experiment the stated errors are exactly correct, a luxury real data never
grant.

In [ ]:
G_full = fault.greens_matrix(station_lat, station_lon)
sigma = 0.01                             # 1 cm of noise on every component
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, 3 * n_stations)

In [ ]:
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=d_obs[0::3], north=d_obs[1::3], up=d_obs[2::3],
    sigma_east=np.full(n_stations, sigma),
    sigma_north=np.full(n_stations, sigma),
    sigma_up=np.full(n_stations, sigma),
)
print(f"{N} patches, {n_stations} stations, {d_obs.size} observations")

## 6. The data we are trying to explain

Here is the true slip we will pretend not to know, and the noisy GNSS
displacements it produced. Our job is to recover the left panel using only the
arrows on the right.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
geodef.plot.slip(fault, slip_true[N:], ax=ax1, coords="geographic",
                 title="True dip slip (unknown to the inversion)",
                 colorbar_label="Slip (m)")
geodef.plot.vectors(gnss, fault, ax=ax2, title="Observed GNSS (with noise)")
plt.tight_layout()

## 7. Solve it with weighted least squares

`geodef.solve` runs the whole weighted least-squares machinery in one call.
Because the true slip is pure dip slip, we solve for a single dip-slip value
per patch with `components="dip"`; Chapter 07 explores the component choices
in detail. We pass no regularization argument, so this is the plain,
unregularized solve.

In [ ]:
raw = geodef.solve(fault, gnss, components="dip")
print(f"reduced chi^2: {raw.reduced_chi2:.3f}")
print(f"RMS residual:  {raw.rms * 1000:.2f} mm")

**Reduced chi-squared** ($\chi^2_\nu$) is the standard fit score. It averages
the squared residuals after dividing each one by its stated uncertainty:

$$ \chi^2_\nu = \frac{1}{\nu}\sum_i
   \left(\frac{d_i - (\text{prediction})_i}{\sigma_i}\right)^2, $$

where $\nu$ counts the degrees of freedom (roughly, observations minus model
parameters). A value near 1 means the typical residual is about the size of
the stated error bar; the model fits the data as well as anyone could ask.

Our value is near or below 1, and the RMS residual is smaller than the 1 cm of
noise we added. By every data-fit measure, this inversion is a success. Now
look at the slip.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
limit_true = slip_true[N:].max()
limit_raw = abs(raw.dip_slip).max()
geodef.plot.slip(fault, slip_true[N:], ax=ax1, cmap="RdBu_r",
                 vmin=-limit_true, vmax=limit_true, title="True slip",
                 colorbar_label="Slip (m)")
geodef.plot.slip(fault, raw.dip_slip, ax=ax2, cmap="RdBu_r",
                 vmin=-limit_raw, vmax=limit_raw, title="Unregularized WLS",
                 colorbar_label="Slip (m)")
plt.tight_layout()

Check the two color scales before comparing the panels; they are very
different. The recovered slip swings between large positive and large negative
values, in an alternating pattern with no resemblance to the smooth 3 m bump
we buried, and no geological meaning: the "negative" patches would be normal
faulting embedded inside a thrust event.

## 8. Diagnosis: overfitting

What happened is called **overfitting**: the model has enough freedom to fit
not only the signal in the data but also the noise, and it used that freedom.
Fitting noise requires wild, oscillating slip, because only wild oscillations
can reproduce the random wiggles the noise added to each station.

A few numbers make the diagnosis concrete. The recovered peak slip is far
larger than the true 3 m, and the patch-to-patch roughness is orders of
magnitude larger than the truth's.

In [ ]:
def roughness(slip_values):
    # typical size of the jump between neighboring patch values
    return float(np.std(np.diff(slip_values)))

truth = slip_true[N:]
print(f"true slip: peak {truth.max():6.2f} m, roughness {roughness(truth):8.3f}")
print(f"recovered: peak {raw.dip_slip.max():6.2f} m, "
      f"roughness {roughness(raw.dip_slip):8.3f}")

The amplifier responsible is the ill-conditioned matrix from Section 4. Its
condition number tells us how much the inversion can magnify noise in the
worst case, and here it is enormous: 1 cm of station noise is more than enough
to produce meters of spurious slip.

In [ ]:
G_dip = G_full[:, N:]                    # dip-slip columns only
condition = np.linalg.cond(G_dip.T @ G_dip)
print(f"condition number of G^T G: {condition:.1e}")

Meanwhile the fit to the data remains excellent, which is exactly what makes
overfitting dangerous. If we only ever looked at data space, this model would
pass every test.

In [ ]:
geodef.plot.fit(gnss.obs, raw.predicted,
                title="Observed vs. predicted: the fit looks perfect")

**Checkpoint.** The inversion fit the data to better than the noise level. Why
is that a warning sign rather than a success?

<details><summary>Answer</summary>

The data contain noise that no true model should be able to reproduce. Fitting
*below* the noise level means the model is reproducing the random part of the
data as well as the signal, which is only possible by adding spurious
structure to the slip.

</details>

## 9. What we actually need

The data cannot distinguish the true smooth model from the wild oscillating
one; both fit within the error bars. Choosing between them requires
information from outside the data, such as "slip on a fault varies smoothly
from patch to patch." Adding that kind of information in a controlled way is
called **regularization**, and it is the subject of Chapter 04.

It is worth pausing on what this failure does *not* mean. Weighted least
squares is not a bad method; it did exactly what we asked. The problem is that
"fit the data best" was an incomplete request, because many very different
slip models fit these data equally well.

## Checkpoint questions

**Why do the recovered slip errors alternate in sign from patch to patch?**

<details><summary>Answer</summary>

Alternating patterns are the slip combinations the surface data are least
sensitive to: the contributions of neighboring patches nearly cancel at the
stations. Because the data barely see these patterns, the inversion can use
them freely, and amplified noise lands there.

</details>

**What does a reduced chi-squared near 1 actually certify?**

<details><summary>Answer</summary>

Only that the residuals are about the size of the stated uncertainties, under
the assumed noise model. It says nothing about whether the slip model is
unique, stable, or geologically reasonable; our nonsense model scored
beautifully.

</details>

**Why is it better to call a solver than to compute
$(G^{\mathsf T}WG)^{-1}$ yourself?**

<details><summary>Answer</summary>

For the same reason Chapter 00 preferred `np.linalg.solve` to `np.linalg.inv`:
dedicated solvers and factorizations are faster and less affected by numerical
roundoff, which matters most precisely when the matrix is ill-conditioned.

</details>

## Common mistakes

- **Trusting the fit statistics alone.** Reduced chi-squared and RMS live
  entirely in data space. An excellent fit is compatible with a meaningless
  model, as this chapter just demonstrated.
- **Interpreting oscillations as geology.** A tight checkerboard of
  alternating slip in an inversion result is almost always amplified noise,
  not a real pattern of asperities.
- **Using uncertainties that omit real error sources.** The weights come from
  the stated $\sigma_i$. If those leave out systematic errors, the weighting
  and every chi-squared value built on it are misleading.
- **Concluding the problem is unsolvable.** The failure here is a failure of
  the *unconstrained* formulation, not of inversion itself. With extra
  information stated openly, the next chapter recovers the slip well.

## Recap

- The honest inverse problem includes noise:
  $\mathbf{d} = G\mathbf{m} + \boldsymbol{\varepsilon}$.
- Weighted least squares divides residuals by their uncertainties and is
  GeoDef's default solve.
- Nearly parallel columns of $G$ make $G^{\mathsf T}WG$ ill-conditioned, and
  its inverse amplifies noise into large, oscillating slip.
- Fit statistics measure data space only; they cannot detect an unstable
  model.

Chapter 04 supplies the missing ingredient: regularization, and the tools for
choosing how much of it to use.

## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are in `solutions/03_unregularized_inversion_solutions.ipynb`.

1. **More noise, more trouble.** Double the noise level (2 cm instead of 1 cm,
   in both the data and the stated uncertainties) and rerun the unregularized
   solve. How do the peak recovered slip and reduced chi-squared change?
2. **Fewer stations.** Remove stations (for example, keep every other one)
   until there are fewer observations than unknowns, and describe what happens
   to the solution along the way.
3. **Whitening by hand.** Divide each row of $G$ and each observation by its
   uncertainty, solve the ordinary least-squares problem with
   `np.linalg.lstsq`, and confirm you reproduce the `geodef.solve` result.
4. **Challenge: see the blind spot.** Compute the singular value decomposition
   of the whitened $G$, plot the right singular vector with the smallest
   singular value as a slip map, and compare its pattern with the oscillations
   in the recovered slip.

## Further reading

- Menke, W. (2018), *Geophysical Data Analysis: Discrete Inverse Theory*,
  4th ed., Chapters 3–4 on least squares and ill-posedness.
- Aster, R., Borchers, B., and Thurber, C. (2018), *Parameter Estimation and
  Inverse Problems*, 3rd ed., Chapters 2–4 on the SVD view of instability.
- Golub, G. H., and Van Loan, C. F. (2013), *Matrix Computations*, 4th ed.,
  for the numerical linear algebra behind the solvers.
- [GeoDef inversion documentation](../docs/invert.md) for the full `solve`
  interface.
- The next course chapter is `04_regularization.ipynb`.